# K1 - Procrustes across 21 pairs in raw space  (tests C.13.7)

Opens with an alignment check against a known-good anchor: bge-SBERT at 152.8x chance confirms the inherited ids for bert and sbert.

**Fits a global scale with the rotation** - without it the 234x row-norm spread made the measure undefined (R-squared as low as -7643). That makes this a similarity transform, so levels are NOT comparable to the rotation-only 0.038 or 0.194; the ORDERING is what it establishes.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# K1 — Procrustes across 21 pairs in RAW space  (tests report C.13.7).
# Removes a stated non-comparability, adds no experiment.
#
# WHY. The report currently carries two Procrustes numbers that cannot be
# put in the same sentence:
#
#   Experiment B   0.038   one pair, raw 512-d into raw 768-d, no PCA
#   C.13.7         0.194   mean over 21 pairs, AFTER common-PCA reduction
#
# and says so explicitly: "systematically higher than, and NOT comparable
# to, Experiment B's raw 0.038." The PCA step equalises per-axis variance
# and therefore performs part of the alignment before Procrustes is scored.
#
# The reduction was used because the spaces have unequal ambient widths
# (512 / 768 / 2048) and orthogonal Procrustes needs a square rotation.
# But Experiment B solved exactly that case without PCA, so the same
# construction extends to all 21 pairs: zero-pad the narrower space to the
# wider width and solve for an orthogonal R there. Padding adds no variance
# and performs no alignment, which is the property common-PCA lacks.
#
# WHAT THIS CHANGES. Nothing about the finding. "Related, not rigid" is
# already confirmed broadly. What changes is that the headline near-isometry
# number and the 21-pair confirmation become the SAME measurement at two
# scopes, so they can be quoted together and the caveat paragraph can go.
#
# PRE-REGISTERED EXPECTATION. Raw-space values should come out LOWER than
# the common-PCA values across the board - that is the whole point of the
# caveat - and the pair ORDER should be preserved. If the order changes,
# the common-PCA ranking in Table C12k was partly an artifact of the
# reduction and the report needs to say so.
#
# Scored as held-out R-squared, matching Table 4, so 0.038 is the number
# this must be comparable to.
#
# DIRECTIONALITY, which the common-PCA version did not have to worry about.
# Zero-padding makes Procrustes ASYMMETRIC: R can only produce outputs
# inside the narrower space's column span, so predicting a 768-d space from
# a padded 384-d one is rank-limited in a way the reverse is not. Common-PCA
# put both spaces at the same width and was therefore near-symmetric, so
# "21 pairs" was well defined. In raw space there are 42 ORDERED pairs.
# Experiment B's 0.038 is specifically narrow -> wide (512 into 768), so
# narrow -> wide is the headline convention here and both directions are
# printed. If the two differ a lot, say so rather than picking the flattering
# one.
# ==========================================================
import os
import numpy as np
from pathlib import Path
from itertools import combinations

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
N_TRAIN, N_EVAL = 8533, 1000
SEED = 0

# --- adjust to your cached-space filenames ----------------------------
SPACES_NPZ = DATA_DIR / "hub_rebuilt.npz"   # raw_* arrays written by G0
# ----------------------------------------------------------------------

z = np.load(SPACES_NPZ, allow_pickle=True)
X = {k[4:]: np.asarray(z[k], dtype=np.float64)
     for k in z.files if k.startswith("raw_")}
names = sorted(X)
assert len(names) == 7, f"expected 7 raw_ spaces, got {names}"
print(f"{len(names)} spaces: " + ", ".join(f"{n}({X[n].shape[1]}d)" for n in names))
for n in names:
    assert X[n].shape[0] == N_TRAIN + N_EVAL, f"{n} row count misaligned"
print("  widths: " + ", ".join(f"{n}={X[n].shape[1]}" for n in names))


def pad_to(A, width):
    """Zero-pad to `width` columns. Adds no variance and no rotation -
    unlike a PCA reduction, which equalises per-axis scale and thereby
    does part of the alignment before Procrustes is ever solved."""
    if A.shape[1] == width:
        return A
    out = np.zeros((A.shape[0], width))
    out[:, :A.shape[1]] = A
    return out


def procrustes_r2(A, B, n_train=N_TRAIN):
    """Orthogonal Procrustes B ~= A R, scored as held-out R-squared.

    Centres on TRAIN means only - centring on the full set would leak
    eval rows into the fit, which is the same error the project's own
    point-in-time discipline exists to prevent.
    """
    w = max(A.shape[1], B.shape[1])
    A, B = pad_to(A, w), pad_to(B, w)
    mu_a, mu_b = A[:n_train].mean(0), B[:n_train].mean(0)
    Ac, Bc = A - mu_a, B - mu_b

    U, Sv, Vt = np.linalg.svd(Ac[:n_train].T @ Bc[:n_train], full_matrices=False)
    R = U @ Vt                                    # orthogonal by construction

    # ONE global scalar alongside the rotation. Without it this measure is
    # undefined across these spaces: orthogonal Procrustes preserves norm
    # (||AR|| = ||A||), and row norms here span 234x (bge 0.89, GPT-2
    # 208.21). Predicting bge from img_base then yields a prediction ~59x
    # too large, the residual dwarfs the target variance, and R^2 collapses
    # to -7643 - measuring the scale mismatch, not the geometry.
    #
    # This is why the original used common-PCA: the reduction was
    # equalising scale. Removing it without replacing that function broke
    # the measure. A similarity transform (rotation + one scalar) restores
    # it while staying far more constrained than a full linear map, and it
    # is the honest formalisation of "related, not rigid".
    s_opt = float(Sv.sum() / (Ac[:n_train] ** 2).sum())

    pred = s_opt * (Ac[n_train:] @ R)
    ss_res = float(((Bc[n_train:] - pred) ** 2).sum())
    ss_tot = float((Bc[n_train:] ** 2).sum())
    return 1.0 - ss_res / ss_tot, s_opt

In [ ]:
# ---------- 0. ALIGNMENT CHECK, before any pair is computed ----------
# Two of the seven spaces - txt_bert and txt_sbert - are standalone files
# with no stored `keep` array. G0 gave them img_base's image ids because
# their row count matches. That is an inference from row count, not a
# stored fact, and if it is wrong those two spaces are shuffled relative
# to everything else. Eleven of the 21 pairs involve them, so half this
# table would be noise while looking entirely normal.
#
# It cannot be checked against the hub: bert and sbert carry 0.17% of the
# concat's energy, so the spectrum gate is blind to their ordering.
#
# It CAN be checked against txt_bge. bge lives inside crossmodal_pairs.npz
# next to its own img array, paired within the file, so its alignment is a
# stored fact rather than an inference. And the report's own G5 result puts
# bge-SBERT at the TOP of the agreement table - the strongest pair in the
# project. So: measure bge-SBERT against an anchor known to be correct. If
# SBERT's rows were shuffled, that pair drops to chance. This is a
# published result being used as a fixture, which costs nothing and tests
# exactly the thing in doubt.

def knn_overlap(A, B, k=10, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(A), min(n, len(A)), replace=False)
    a = A[idx] / (np.linalg.norm(A[idx], axis=1, keepdims=True) + 1e-8)
    b = B[idx] / (np.linalg.norm(B[idx], axis=1, keepdims=True) + 1e-8)
    na = np.argsort(-(a @ a.T), axis=1)[:, 1:k + 1]
    nb = np.argsort(-(b @ b.T), axis=1)[:, 1:k + 1]
    return float(np.mean([len(set(x) & set(y)) / k for x, y in zip(na, nb)]))

print("=" * 62)
print("STEP 0 - alignment check against a known-good anchor")
print("=" * 62)
chance = 10 / 2000
pairs_to_check = [("txt_bge", "txt_sbert"), ("txt_bge", "txt_bert"),
                  ("txt_bert", "txt_sbert"), ("img_base", "img_large")]
ok = True
for a, b in pairs_to_check:
    if a not in X or b not in X:
        continue
    ov = knn_overlap(X[a], X[b])
    ratio = ov / chance
    flag = ""
    if a == "txt_bge" and b == "txt_sbert":
        flag = "  <- report puts this pair TOP; must be well above chance"
        if ratio < 10:
            ok = False
    print(f"  {a:10s} - {b:10s}  overlap@10 {ov:.3f}  = {ratio:6.1f}x chance{flag}")

if not ok:
    print("\n  ALIGNMENT FAILED. bge-SBERT is near chance, but the report has")
    print("  it as the strongest pair in the project. bge's alignment is")
    print("  guaranteed by file construction, so SBERT is the one that is")
    print("  shuffled - which means txt_bert almost certainly is too, since")
    print("  both were given inherited ids the same way.")
    print("  STOPPING: 11 of the 21 pairs below would be noise.")
    raise SystemExit("alignment check failed")
print("\n  Alignment consistent with the published G5 ordering. Proceeding.")

In [ ]:
# ---------- all ordered-pair-free combinations ----------
rows = []
for a, b in combinations(names, 2):
    # order by ambient width so the headline is narrow -> wide, as in Exp B
    lo, hi = (a, b) if X[a].shape[1] <= X[b].shape[1] else (b, a)
    fwd, s_f = procrustes_r2(X[lo], X[hi])   # narrow -> wide, as in Exp B
    rev, _ = procrustes_r2(X[hi], X[lo])     # wide -> narrow, for the gap
    rows.append((f"{lo} -> {hi}", fwd, rev, abs(fwd - rev), s_f))

rows.sort(key=lambda t: -t[1])
vals = np.array([r[1] for r in rows])
revs = np.array([r[2] for r in rows])
asym = np.array([r[3] for r in rows])
scales = np.array([r[4] for r in rows])

print("\n" + "=" * 62)
print("Procrustes in RAW space, held-out R-squared (no PCA reduction)")
print("=" * 62)
print(f"  {'ordered pair (narrow -> wide)':<34} {'fwd':>7} {'rev':>8} "
      f"{'gap':>7} {'scale':>9}")
for name, fwd, rev, d, sc in rows:
    print(f"  {name:<34} {fwd:+7.3f} {rev:+8.3f} {d:7.3f} {sc:9.3f}")
print("-" * 62)
print(f"  {'mean, narrow -> wide (headline)':<34} {vals.mean():+7.3f}")
print(f"  {'mean, wide -> narrow':<34} {revs.mean():+7.3f}")
print(f"  {'median, narrow -> wide':<34} {np.median(vals):+7.3f}")
print(f"  {'negative pairs (fwd)':<34} "
      f"{int((vals < 0).sum())} of {len(vals)}")
print(f"  {'max direction gap':<34} {asym.max():7.3f}")

if asym.max() > 0.05:
    print("\n  NOTE: the two directions differ by more than 0.05 on at least")
    print("  one pair. Padding makes this measure asymmetric. Report the")
    print("  narrow -> wide column (comparable to Exp B) and state that the")
    print("  reverse direction differs - do not average them into one number.")

print("\n  reference points from the report:")
print(f"    Experiment B, raw, one pair              +0.038   <- now comparable")
print(f"    C.13.7 mean, after common-PCA            +0.194")
print(f"    ridge (full linear), for contrast        +0.667")

print("\n" + "=" * 62)
if vals.min() < -1.0:
    print("TABLE INVALID. At least one held-out R^2 is below -1, which means")
    print("the fit is worse than predicting the target mean by a large")
    print("factor - a scale failure, not a geometry result. Do not read the")
    print("mean, and do not quote any row. Diagnose before interpreting:")
    print(f"  worst value {vals.min():.1f} on "
          f"{rows[int(np.argmin(vals))][0]}")
    print("  fitted scales span "
          f"{scales.min():.3g} to {scales.max():.3g}")
elif vals.mean() < 0.194:
    print("VERDICT: raw-space values are lower, as the caveat predicted. The")
    print("common-PCA reduction was indeed doing part of the alignment. The")
    print("21-pair result can now be quoted alongside Experiment B's 0.038")
    print("without a comparability warning, and the caveat paragraph in")
    print("C.13.7 can be replaced by this table.")
else:
    print("VERDICT: raw-space values are NOT lower. That contradicts the")
    print("stated reason for the caveat and needs explaining before either")
    print("number is used. Do not quote them together until it is.")

print("\nCheck the ORDER against Table C12k before rewriting anything:")
print("  if bge-SBERT still tops the table and cross-modal / GPT-2 pairs")
print("  still go negative, the ranking was not an artifact of the PCA")
print("  step and every downstream claim about pair ordering stands.")
print("  If the order moved, that is a finding about C12k, not a footnote.")